In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.linalg import logm, expm
import warnings
import matplotlib.pyplot as plt

In [2]:
filename = "data/rc_kernel.txt"
df = pd.read_csv(filename, sep='\t', header=None)
df.columns = range(df.shape[1])
n=8
n_elem = df.shape[1]
n = int((-1 + np.sqrt(1 + 8 * n_elem)) / 2)
assert n * (n + 1) // 2 == n_elem
tril_idx = np.tril_indices(n)
df.head(10)

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,0.000362,0.000308,0.000446,0.000370,0.000368,0.000711,0.000251,0.000252,0.000294,0.000236,...,0.000053,0.000141,0.000246,0.000247,0.000270,0.000217,0.000210,0.000108,0.000124,0.000257
1,0.000639,0.000501,0.000738,0.000490,0.000380,0.000539,0.000394,0.000306,0.000393,0.000420,...,0.000115,0.000220,0.000376,0.000320,0.000318,0.000300,0.000267,0.000097,0.000078,0.000337
2,0.000534,0.000472,0.000762,0.000427,0.000492,0.000500,0.000343,0.000383,0.000359,0.000325,...,0.000224,0.000397,0.000337,0.000469,0.000384,0.000321,0.000295,0.000189,0.000351,0.000408
3,0.000410,0.000366,0.000822,0.000207,0.000276,0.000368,0.000142,0.000242,0.000240,0.000225,...,0.000052,0.000147,0.000117,0.000217,0.000216,0.000191,0.000183,0.000042,0.000047,0.000215
4,0.000584,0.000493,0.000612,0.000406,0.000365,0.000476,0.000298,0.000223,0.000265,0.000261,...,0.000040,0.000094,0.000358,0.000269,0.000296,0.000294,0.000305,0.000183,0.000073,0.000414
5,0.000481,0.000527,0.000779,0.000490,0.000614,0.000729,0.000468,0.000515,0.000549,0.000558,...,0.000171,0.000247,0.000493,0.000556,0.000595,0.000586,0.000510,0.000331,0.000278,0.000652
6,0.000430,0.000283,0.000401,0.000394,0.000346,0.000700,0.000298,0.000205,0.000312,0.000243,...,0.000057,0.000229,0.000331,0.000244,0.000381,0.000258,0.000257,0.000098,0.000052,0.000326
7,0.000612,0.000620,0.000786,0.000740,0.000746,0.001307,0.000435,0.000448,0.000502,0.000407,...,0.000104,0.000279,0.000506,0.000523,0.000683,0.000415,0.000454,0.000201,0.000127,0.000540
8,0.000462,0.000539,0.000754,0.000436,0.000524,0.000578,0.000334,0.000453,0.000373,0.000325,...,0.000235,0.000483,0.000380,0.000486,0.000452,0.000358,0.000352,0.000259,0.000370,0.000466
9,0.001263,0.001057,0.001053,0.002043,0.001784,0.004391,0.000966,0.000865,0.001749,0.000849,...,0.000121,0.000243,0.001003,0.000866,0.001805,0.000809,0.000640,0.000274,0.000279,0.000908


In [3]:
def vech_to_sym(row):
    M = np.zeros((n, n))
    M[tril_idx] = row
    return M + M.T - np.diag(np.diag(M))

def mat_to_vech(M):
    return M[tril_idx]

def nearest_psd(A, eps=1e-8):
    A = (A + A.T) / 2
    w, v = np.linalg.eigh(A)
    return v @ np.diag(np.maximum(w, eps)) @ v.T

In [4]:
def run_har(df_transformed, n_test=600):
    RC_d = df_transformed.shift(1)
    RC_w = df_transformed.rolling(5).mean().shift(1)
    RC_m = df_transformed.rolling(22).mean().shift(1)
    X = pd.concat([RC_d, RC_w, RC_m], axis=1).dropna()
    y = df_transformed.loc[X.index]

    X_train, y_train = X.iloc[:-n_test], y.iloc[:-n_test]
    X_test,  y_test  = X.iloc[-n_test:], y.iloc[-n_test:]

    models = {}
    for col in y_train.columns:
        res = sm.OLS(y_train[col], sm.add_constant(X_train)).fit()
        models[col] = res

    X_test_c = sm.add_constant(X_test)
    y_pred = pd.DataFrame(
        {col: models[col].predict(X_test_c) for col in y_test.columns},
        index=y_test.index
    )
    return models, y_pred, y_test

In [5]:
def df_to_logm(df):
    out = np.zeros_like(df.values, dtype=float)
    for t in range(len(df)):
        try:
            S = logm(nearest_psd(vech_to_sym(df.iloc[t].values)))
            S = np.real(S)
            S = (S + S.T) / 2
            out[t] = mat_to_vech(S)
        except Exception as e:
            warnings.warn(f"logm failed at t={t}: {e}")
            out[t] = out[t - 1] if t > 0 else np.zeros(n_elem)
    return pd.DataFrame(out, index=df.index, columns=df.columns)

In [6]:
def chol_pred_to_rc(row):
    L = np.zeros((n, n))
    L[tril_idx] = row
    return L @ L.T

def logm_pred_to_rc(row):
    S = np.zeros((n, n))
    S[tril_idx] = row
    S = S + S.T - np.diag(np.diag(S))
    return expm(S)

In [7]:
#normal not psd
_, pred_har, y_test_har = run_har(df)



In [8]:
#cholesky:
df_chol = pd.read_csv("data/cholesky_kernel.txt", sep='\t', header=None)
df_chol.columns = range(df_chol.shape[1])

_, pred_chol_transformed, _ = run_har(df_chol)

pred_chol_rc = np.stack([
    chol_pred_to_rc(pred_chol_transformed.iloc[t].values)
    for t in range(len(pred_chol_transformed))
])

In [9]:
#logm exp
df_log = df_to_logm(df)
_, pred_log_transformed, y_test_log = run_har(df_log)

pred_log_rc = np.stack([
    logm_pred_to_rc(pred_log_transformed.iloc[t].values)
    for t in range(len(pred_log_transformed))
])

In [10]:
pred_har.head()

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
3392,0.000150,0.000197,0.000404,0.000114,0.000193,0.000118,0.000114,0.000175,0.000125,0.000122,...,0.000034,0.000059,0.000166,0.000214,0.000169,0.000161,0.000223,0.000053,0.000033,0.000323
3393,0.000143,0.000198,0.000541,0.000089,0.000209,0.000103,0.000112,0.000197,0.000119,0.000128,...,0.000062,0.000099,0.000173,0.000228,0.000154,0.000169,0.000217,0.000057,0.000030,0.000310
3394,0.000157,0.000122,0.000320,0.000050,0.000096,0.000059,0.000076,0.000088,0.000058,0.000105,...,0.000051,0.000094,0.000118,0.000106,0.000079,0.000101,0.000157,0.000032,-0.000018,0.000219
3395,0.000279,0.000197,0.000453,0.000104,0.000114,-0.000022,0.000162,0.000149,0.000067,0.000224,...,0.000119,0.000230,0.000169,0.000149,0.000084,0.000154,0.000196,0.000081,0.000049,0.000241
3396,0.000283,0.000217,0.000418,0.000151,0.000175,0.000209,0.000170,0.000143,0.000137,0.000192,...,0.000109,0.000231,0.000181,0.000160,0.000153,0.000151,0.000211,0.000077,0.000049,0.000274


In [11]:
def check_psd(pred_df, label=""):
    n_violations = 0
    for t in range(len(pred_df)):
        M = vech_to_sym(pred_df.iloc[t].values)
        min_eig = np.linalg.eigvalsh(M).min()
        if min_eig < 0:
            n_violations += 1
    print(f"{label}: {n_violations}/{len(pred_df)} non-PSD ({100*n_violations/len(pred_df):.1f}%)")

check_psd(pred_har, "HAR (plain)")


HAR (plain): 445/600 non-PSD (74.2%)


In [12]:
n_viol_chol = sum(
    np.linalg.eigvalsh(pred_chol_rc[t]).min() < 0
    for t in range(len(pred_chol_rc))
)

print(f"HAR-Cholesky recovered RC: {n_viol_chol} non-PSD")


HAR-Cholesky recovered RC: 0 non-PSD


In [13]:
n_viol_log = sum(
    np.linalg.eigvalsh(pred_log_rc[t]).min() < 0
    for t in range(len(pred_log_rc))
)
print(f"HAR-LogExp   recovered RC: {n_viol_log} non-PSD")

HAR-LogExp   recovered RC: 0 non-PSD


In [14]:
realized_rc = np.stack([
    vech_to_sym(y_test_har.iloc[t].values)
    for t in range(len(y_test_har))
])

pred_har_rc = np.stack([
    vech_to_sym(pred_har.iloc[t].values)
    for t in range(len(pred_har))
])


predictions = {
    "HAR":          pred_har_rc,
    "HAR-Cholesky": pred_chol_rc,
    "HAR-LogExp":   pred_log_rc,
}

for name, pred in predictions.items():
    print(f"{name}: {pred.shape}, realized: {realized_rc.shape}")

HAR: (600, 8, 8), realized: (600, 8, 8)
HAR-Cholesky: (600, 8, 8), realized: (600, 8, 8)
HAR-LogExp: (600, 8, 8), realized: (600, 8, 8)


In [15]:
def save_rc_predictions(pred_rc, filename):
    rows = []
    for t in range(len(pred_rc)):
        rows.append(mat_to_vech(pred_rc[t]))
    df_out = pd.DataFrame(rows)
    df_out.to_csv(filename, sep='\t', index=False, header=False)
    print(f"Saved to {filename}")

save_rc_predictions(pred_chol_rc,  "data/pred_chol_kernel.txt")
save_rc_predictions(pred_log_rc,   "data/pred_log_kernel.txt")

Saved to data/pred_chol_kernel.txt
Saved to data/pred_log_kernel.txt
